# Models Training Code
This notebook trains Spark ML models for predicting taxi trip fare.


# Setup


## Imports
This cell imports Spark, Spark ML, metrics, and small filesystem utilities used later.

In [1]:
# Import packages for Spark, ML, metrics, filesystem operations, and time encoding.
import math
import os
import shutil
import subprocess

from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import Imputer, OneHotEncoder
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import DecisionTreeRegressor
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.tuning import CrossValidator
from pyspark.ml.tuning import ParamGridBuilder
from pyspark.sql import SparkSession
from pyspark.sql import functions as F


## Spark Session
The session runs on YARN, enables Hive support, and uses the team Hive warehouse.

In [2]:
# Team number and Hive warehouse path are used by the Spark session.
team = 27
warehouse = "project/hive/warehouse"

# Spark runs on YARN and connects to Hive metastore to read project tables.
spark = SparkSession.builder\
        .appName("{} - spark ML".format(team))\
        .master("yarn")\
        .config("hive.metastore.uris", "thrift://hadoop-02.uni.innopolis.ru:9883")\
        .config("spark.sql.warehouse.dir", warehouse)\
        .config("spark.sql.avro.compression.codec", "snappy")\
        .enableHiveSupport()\
        .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/10 21:46:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/10 21:46:53 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/05/10 21:46:53 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/05/10 21:46:54 WARN DomainSocketFactory: The short-circuit local reads feature cannot be used because libhadoop cannot be loaded.
26/05/10 21:46:55 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


# Initial Data Analysis


## Hive Checks
These cells confirm that the database and source table are available before modeling.

In [3]:
# Show available Hive databases.
spark.sql("SHOW DATABASES").show()


+--------------------+
|           namespace|
+--------------------+
|             default|
|           ml_stage3|
|             retake1|
|             root_db|
|                show|
|    team00_projectdb|
|          team0_dbms|
|     team0_projectdb|
|             team0db|
|            team0db1|
|            team0db2|
|    team11_projectdb|
|           team12_db|
|team12_hive_proje...|
|    team12_projectdb|
|    team13_projectdb|
|team13_projectdb_...|
|    team14_projectdb|
|    team15_projectdb|
|    team16_projectdb|
+--------------------+
only showing top 20 rows



In [4]:
# Use the team project database.
spark.sql("USE team27_projectdb")


DataFrame[]

In [5]:
# List tables and verify that fact_trips_raw exists.
spark.sql("SHOW TABLES").show(100, False)


+----------------+----------------------+-----------+
|namespace       |tableName             |isTemporary|
+----------------+----------------------+-----------+
|team27_projectdb|books                 |false      |
|team27_projectdb|fact_trips_raw        |false      |
|team27_projectdb|q10_results           |false      |
|team27_projectdb|q11_results           |false      |
|team27_projectdb|q12_results           |false      |
|team27_projectdb|q13_results           |false      |
|team27_projectdb|q14_results           |false      |
|team27_projectdb|q15_results           |false      |
|team27_projectdb|q1_results            |false      |
|team27_projectdb|q2_results            |false      |
|team27_projectdb|q3_results            |false      |
|team27_projectdb|q4_results            |false      |
|team27_projectdb|q5_results            |false      |
|team27_projectdb|q6_results            |false      |
|team27_projectdb|q7_results            |false      |
|team27_projectdb|q8_results

In [6]:
# Inspect sample records from the source table.
spark.sql("SELECT * FROM fact_trips_raw LIMIT 10").show()


+-------+-------+------------+------------+--------------------+------------------+------------+----------+-------------------+--------------------+---------------------+----------------------+-----+----+-----+------+----------+--------------------+---------------------+---------------------+----------------------+---------+----------+--------+---------+------------+----------------------+-----------------+----------------+
|trip_id|taxi_id|company_code|payment_type|trip_start_timestamp|trip_end_timestamp|trip_seconds|trip_miles|pickup_census_tract|dropoff_census_tract|pickup_community_area|dropoff_community_area| fare|tips|tolls|extras|trip_total|pickup_latitude_code|pickup_longitude_code|dropoff_latitude_code|dropoff_longitude_code|trip_year|trip_month|trip_day|trip_hour|trip_weekday|is_valid_end_timestamp|is_valid_duration|is_valid_amounts|
+-------+-------+------------+------------+--------------------+------------------+------------+----------+-------------------+-----------------

## Modeling
This is the main section of this notebook here we will preprocess features and build our models

### Feature Preprocessing
Before moderling we firstly need to create embeddings of our rows, so our models could use them

## Feature Lists
The model uses trip duration, distance, time, payment type, community areas, and validation fields. High-cardinality identifiers and fine location grids are dropped to reduce overfitting.

In [7]:
# Feature groups used by the preprocessing pipeline.
CATEGORICAL_COLS = [
    "payment_type",
    "pickup_community_area",
    "dropoff_community_area",
]
NUMERIC_COLS = ["trip_seconds", "trip_miles", "trip_year"]
BOOLEAN_COLS = ["is_valid_end_timestamp", "is_valid_duration", "is_valid_amounts"]
CYCLE_COLS = [
    "trip_month_sin",
    "trip_month_cos",
    "trip_day_sin",
    "trip_day_cos",
    "trip_hour_sin",
    "trip_hour_cos",
    "trip_weekday_sin",
    "trip_weekday_cos",
]

for path in ["../data", "../models", "../output"]:
    os.makedirs(path, exist_ok=True)


## Load And Clean Data
This step reads `fact_trips_raw`, creates the target column, converts types, handles null categorical values, and encodes cyclic date/time parts.

In [8]:
# Read the Hive table and create the regression label from fare.
data = spark.table("team27_projectdb.fact_trips_raw")
data = data.withColumn("label", F.col("fare").cast("double"))

# Cast numeric columns to doubles for Spark ML.
for col_name in NUMERIC_COLS:
    data = data.withColumn(col_name, F.col(col_name).cast("double"))

# Convert boolean flags to numeric 0/1 features.
for col_name in BOOLEAN_COLS:
    data = data.withColumn(col_name, F.when(F.col(col_name).cast("boolean"), 1.0).otherwise(0.0))

# Convert categorical fields to strings and keep missing values as a category.
for col_name in CATEGORICAL_COLS:
    data = data.withColumn(col_name, F.coalesce(F.col(col_name).cast("string"), F.lit("missing")))

# Encode cyclic time parts with sine and cosine values.
month_value = 2.0 * math.pi * F.col("trip_month").cast("double") / F.lit(12.0)
day_value = 2.0 * math.pi * F.col("trip_day").cast("double") / F.lit(31.0)
hour_value = 2.0 * math.pi * F.col("trip_hour").cast("double") / F.lit(24.0)
weekday_value = 2.0 * math.pi * F.col("trip_weekday").cast("double") / F.lit(7.0)

data = data.withColumn("trip_month_sin", F.sin(month_value)).withColumn("trip_month_cos", F.cos(month_value))
data = data.withColumn("trip_day_sin", F.sin(day_value)).withColumn("trip_day_cos", F.cos(day_value))
data = data.withColumn("trip_hour_sin", F.sin(hour_value)).withColumn("trip_hour_cos", F.cos(hour_value))
data = data.withColumn("trip_weekday_sin", F.sin(weekday_value)).withColumn("trip_weekday_cos", F.cos(weekday_value))

# Keep only model inputs and remove rows without a valid positive label.
data = data.select(["label"] + NUMERIC_COLS + BOOLEAN_COLS + CYCLE_COLS + CATEGORICAL_COLS)
data = data.where((F.col("label").isNotNull()) & (F.col("label") > 0.0)).cache()
data = data.orderBy(F.rand(42)).limit(1_000_000).cache()
print("Sampled records:", data.count())

data.printSchema()
data.limit(10).show()

Sampled records: 1000000
root
 |-- label: double (nullable = true)
 |-- trip_seconds: double (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_year: double (nullable = true)
 |-- is_valid_end_timestamp: double (nullable = false)
 |-- is_valid_duration: double (nullable = false)
 |-- is_valid_amounts: double (nullable = false)
 |-- trip_month_sin: double (nullable = true)
 |-- trip_month_cos: double (nullable = true)
 |-- trip_day_sin: double (nullable = true)
 |-- trip_day_cos: double (nullable = true)
 |-- trip_hour_sin: double (nullable = true)
 |-- trip_hour_cos: double (nullable = true)
 |-- trip_weekday_sin: double (nullable = true)
 |-- trip_weekday_cos: double (nullable = true)
 |-- payment_type: string (nullable = false)
 |-- pickup_community_area: string (nullable = false)
 |-- dropoff_community_area: string (nullable = false)

+-----+------------+----------+---------+----------------------+-----------------+----------------+-------------------+------------

## Preprocessing Pipeline
The split happens before fitting preprocessing. The pipeline learns categorical mappings, imputes numeric values, and assembles all features without scaling them.

In [9]:
# Split data before fitting preprocessing to avoid test-data leakage.
train_data, test_data = data.randomSplit([0.8, 0.2], seed=42)

indexed_cols = [col_name + "_idx" for col_name in CATEGORICAL_COLS]
encoded_cols = [col_name + "_vec" for col_name in CATEGORICAL_COLS]
numeric_inputs = NUMERIC_COLS + BOOLEAN_COLS + CYCLE_COLS
imputed_cols = [col_name + "_imp" for col_name in numeric_inputs]

# StringIndexer and OneHotEncoder convert categorical values to vectors.
indexers = [
    StringIndexer(inputCol=col_name, outputCol=idx, handleInvalid="keep")
    for col_name, idx in zip(CATEGORICAL_COLS, indexed_cols)
]
encoder = OneHotEncoder(inputCols=indexed_cols, outputCols=encoded_cols)

# Imputer fills numeric nulls, VectorAssembler creates final features.
imputer = Imputer(inputCols=numeric_inputs, outputCols=imputed_cols).setStrategy("median")
assembler = VectorAssembler(inputCols=imputed_cols + encoded_cols, outputCol="features", handleInvalid="keep")

feature_pipeline = Pipeline(stages=indexers + [encoder, imputer, assembler])
feature_model = feature_pipeline.fit(train_data)

train_ready = feature_model.transform(train_data).select("features", "label")
test_ready = feature_model.transform(test_data).select("features", "label")

train_count = train_data.count()
test_count = test_data.count()
print(f"Training set size: {train_count:,}  |  Test set size: {test_count:,}")

train_ready.show(5, truncate=False)
test_ready.show(5, truncate=False)

Training set size: 800,368  |  Test set size: 199,632


+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|features                                                                                                                                                                                                                              |label|
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|(177,[2,3,5,6,7,8,9,10,11,12,13,16,21,99],[2016.0,1.0,1.0,-0.8660254037844386,0.5000000000000001,-0.8978045395707417,-0.44039415155763423,-0.9659258262890684,0.2588190451025203,-0.9749279121818236,-0.2225209339563146,1.0,1.0,1.0])|0.01 |
|(177,[2,3,5,6,7,8,9,10,11,12,13,14,21,99],[

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|features                                                                                                                                                                                                                              |label|
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|(177,[2,3,5,6,7,8,9,10,11,12,13,14,26,103],[2016.0,1.0,1.0,-0.8660254037844386,0.5000000000000001,-0.5712682150947924,0.8207634412072763,-0.9659258262890684,0.2588190451025203,-0.9749279121818236,-0.2225209339563146,1.0,1.0,1.0]) |0.01 |
|(177,[2,3,5,6,7,8,9,10,11,12,13,14,21,99],[

### Models
Three model families are used: regularized linear regression, random forest regression, and decision tree regression.

## Model Setup
Three regressors are configured with two model-specific hyperparameters each for grid search.

In [10]:
# RMSE is used for hyperparameter selection; MAE and R2 are reported too.
evaluator = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
rmse_evaluator = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
mae_evaluator = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="mae")
r2_evaluator = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="r2")

# First model: strongly regularized linear regression.
lr = LinearRegression(labelCol="label", featuresCol="features", regParam=1.0, elasticNetParam=0.5)
lr_grid = (
    ParamGridBuilder()
    .addGrid(lr.regParam, [1.0]) # [0.1, 1.0]
    .addGrid(lr.elasticNetParam, [1.0]) # [0.5, 1.0]
    .build()
)

# Second model: random forest regression.
rf = RandomForestRegressor(labelCol="label", featuresCol="features", seed=42)
rf_grid = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, [20]) # [20, 50]
    .addGrid(rf.maxDepth, [5]) # [5, 10]
    .build()
)

# Third model: decision tree regression.
dt = DecisionTreeRegressor(labelCol="label", featuresCol="features", seed=42)
dt_grid = (
    ParamGridBuilder()
    .addGrid(dt.maxDepth, [5]) # [5, 10]
    .addGrid(dt.minInstancesPerNode, [1]) # [1, 5]
    .build()
)


### Training
Default models are evaluated first, then cross-validation selects the best tuned model of each type.

## Linear Regression
A default model is evaluated first, then cross-validation chooses the best regularization settings.

In [11]:
# Train a baseline linear regression model without grid search.
lr_default_model = lr.fit(train_ready)
lr_default_predictions = lr_default_model.transform(test_ready)
lr_default_rmse = rmse_evaluator.evaluate(lr_default_predictions)
lr_default_mae = mae_evaluator.evaluate(lr_default_predictions)
lr_default_r2 = r2_evaluator.evaluate(lr_default_predictions)
print("LinearRegression default RMSE={:.4f}, MAE={:.4f}, R2={:.4f}".format(lr_default_rmse, lr_default_mae, lr_default_r2))

# Tune linear regression on train data only.
print("Starting CV for Linear Regression model")
lr_cv = CrossValidator(
    estimator=lr,
    estimatorParamMaps=lr_grid,
    evaluator=evaluator,
    numFolds=3,
    parallelism=2,
    seed=42,
)
lr_cv_model = lr_cv.fit(train_ready)
model1 = lr_cv_model.bestModel

# Evaluate the best linear regression model on test data.
print("Evaluating best Linear Regression model")
predictions1 = model1.transform(test_ready)
metrics1 = {
    "rmse": rmse_evaluator.evaluate(predictions1),
    "mae": mae_evaluator.evaluate(predictions1),
    "r2": r2_evaluator.evaluate(predictions1),
}
metrics1

LinearRegression default RMSE=16.5747, MAE=5.0613, R2=0.2477
Starting CV for Linear Regression model


Evaluating best Linear Regression model


{'rmse': 16.721425791474346,
 'mae': 5.486410601798478,
 'r2': 0.23433110202063578}

## Random Forest
A default forest is evaluated first, then cross-validation tunes tree count and depth.

In [12]:
# Train a baseline random forest model without grid search.
rf_default_model = rf.fit(train_ready)
rf_default_predictions = rf_default_model.transform(test_ready)
rf_default_rmse = rmse_evaluator.evaluate(rf_default_predictions)
rf_default_mae = mae_evaluator.evaluate(rf_default_predictions)
rf_default_r2 = r2_evaluator.evaluate(rf_default_predictions)
print("RandomForestRegressor default RMSE={:.4f}, MAE={:.4f}, R2={:.4f}".format(rf_default_rmse, rf_default_mae, rf_default_r2))

# Tune random forest on train data only.
print("Starting CV for Random Forest model")
rf_cv = CrossValidator(
    estimator=rf,
    estimatorParamMaps=rf_grid,
    evaluator=evaluator,
    numFolds=3,
    parallelism=2,
    seed=42,
)
rf_cv_model = rf_cv.fit(train_ready)
model2 = rf_cv_model.bestModel

# Evaluate the best random forest model on test data.
print("Evaluating best Random Forest model")
predictions2 = model2.transform(test_ready)
metrics2 = {
    "rmse": rmse_evaluator.evaluate(predictions2),
    "mae": mae_evaluator.evaluate(predictions2),
    "r2": r2_evaluator.evaluate(predictions2),
}
metrics2

RandomForestRegressor default RMSE=15.2996, MAE=3.1501, R2=0.3590
Starting CV for Random Forest model


Evaluating best Random Forest model


{'rmse': 15.299644505183773,
 'mae': 3.1501461072472052,
 'r2': 0.35900140994833996}

## Decision Tree
A default decision tree is evaluated first, then cross-validation tunes tree depth and minimum samples per node.

In [13]:
# Train a baseline decision tree model without grid search.
dt_default_model = dt.fit(train_ready)
dt_default_predictions = dt_default_model.transform(test_ready)
dt_default_rmse = rmse_evaluator.evaluate(dt_default_predictions)
dt_default_mae = mae_evaluator.evaluate(dt_default_predictions)
dt_default_r2 = r2_evaluator.evaluate(dt_default_predictions)
print("DecisionTreeRegressor default RMSE={:.4f}, MAE={:.4f}, R2={:.4f}".format(dt_default_rmse, dt_default_mae, dt_default_r2))

# Tune decision tree on train data only.
print("Starting CV for Decision Tree model")
dt_cv = CrossValidator(
    estimator=dt,
    estimatorParamMaps=dt_grid,
    evaluator=evaluator,
    numFolds=3,
    parallelism=2,
    seed=42,
)
dt_cv_model = dt_cv.fit(train_ready)
model3 = dt_cv_model.bestModel

# Evaluate the best decision tree model on test data.
print("Evaluating best Decision Tree model")
predictions3 = model3.transform(test_ready)
metrics3 = {
    "rmse": rmse_evaluator.evaluate(predictions3),
    "mae": mae_evaluator.evaluate(predictions3),
    "r2": r2_evaluator.evaluate(predictions3),
}
metrics3

DecisionTreeRegressor default RMSE=15.2925, MAE=3.0196, R2=0.3596
Starting CV for Decision Tree model


Evaluating best Decision Tree model


{'rmse': 15.292542481889942,
 'mae': 3.019630134958062,
 'r2': 0.35959636891199154}

## Evaluation

## Final Comparison
The final CSV compares tuned models by RMSE, MAE, and R2 on the same test split.

In [14]:
# Compare all three tuned models on the same held-out test set.
rows = [
    ["LinearRegression", metrics1["rmse"], metrics1["mae"], metrics1["r2"]],
    ["RandomForestRegressor", metrics2["rmse"], metrics2["mae"], metrics2["r2"]],
    ["DecisionTreeRegressor", metrics3["rmse"], metrics3["mae"], metrics3["r2"]],
]
evaluation = spark.createDataFrame(rows, ["model", "RMSE", "MAE", "R2"])
evaluation.show(truncate=False)

+---------------------+------------------+------------------+-------------------+
|model                |RMSE              |MAE               |R2                 |
+---------------------+------------------+------------------+-------------------+
|LinearRegression     |16.721425791474346|5.486410601798478 |0.23433110202063578|
|RandomForestRegressor|15.299644505183773|3.1501461072472052|0.35900140994833996|
|DecisionTreeRegressor|15.292542481889942|3.019630134958062 |0.35959636891199154|
+---------------------+------------------+------------------+-------------------+



In [16]:
spark.stop()